In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
def ler_ultima_particao_tabela_spark(spark, source_table):
  """
  Essa função ler a ultima partição das Tabelas no formato delta baaseado na coluna de data_processamento
  """
  try: 
    # Mais performatica para pegar os metadados
    show_partitions_df = spark.sql(f"SHOW PARTITIONS {source_table}")
    # maior partição da data_processamento
    max_partition = show_partitions_df.agg(f.max("data_processamento")).collect()[0][0]
    print(f"partição maxima {max_partition}")

    # pegar o dataframe com maior partição
    return spark.table(f"{source_table}")\
                      .filter(f.col("data_processamento") == max_partition)
  except Exception as e:
    print(f"Erro ao ler o caminho {source_table}: {e}")
    return None


### 1. gold_fato_diario

In [0]:
gold_fato_diario  = ler_ultima_particao_tabela_spark(spark,  "workspace.case_spark_cvm.gold_fato_diario")

In [0]:
display(gold_fato_diario)

In [0]:
gold_fato_diario = gold_fato_diario\
    .withColumn(
        "ano_mes",
        f.to_date(f.date_trunc("month", f.col("dt_comptc")))
    )

### 2. Agregação de colunas

In [0]:
df_agg =  gold_fato_diario\
    .groupBy(
        "cnpj_fundo_classe",
        "ano_mes"
    )\
    .agg(
        f.avg("vl_patrim_liq").alias("pl_medio_mes"),
        f.sum("captc_dia").alias("captacao_bruta_mes"),
        f.sum("resg_dia").alias("resgate_mes"),
        (f.sum("captc_dia") - f.sum("resg_dia")).alias("captacao_liquida_mes"),
    )

### 3. Dados do Ultimo Mes

In [0]:
window_ultimo_dia = Window.partitionBy("cnpj_fundo_classe", "ano_mes").orderBy(f.col("dt_comptc").desc())

df_ultimo_dia = gold_fato_diario\
    .withColumn("rn", f.row_number().over(window_ultimo_dia))\
    .filter(f.col("rn") == 1)\
    .select(
        "cnpj_fundo_classe",
        "ano_mes",
        f.col("vl_patrim_liq").alias("pl_ultimo_dia_mes"),
        f.col("nr_cotst").alias("nr_cotistas_ultimo_dia")
    )

### 4. Join

In [0]:
gold_cubo_captacao_pl = df_agg\
  .join(
    df_ultimo_dia,
    on=["cnpj_fundo_classe", "ano_mes"],
    how="inner"
  )

### 5. Variação de cotista

In [0]:
window_lag_mes = Window.partitionBy("cnpj_fundo_classe").orderBy("ano_mes")

gold_cubo_captacao_pl = gold_cubo_captacao_pl\
  .withColumn(
    "cotistas_mes_anterior",
    f.lag("nr_cotistas_ultimo_dia").over(window_lag_mes)
  )\
  .withColumn(
    "variacao_cotistas_mes",
    f.when(f.col("cotistas_mes_anterior").isNotNull(),
           f.col("nr_cotistas_ultimo_dia") - f.col("cotistas_mes_anterior"))
    .otherwise(0)
  )\
  .withColumn(
    "flag_captacao_negativa",
    f.when(f.col("captacao_liquida_mes") <0, f.lit("S"))
    .otherwise(f.lit("N"))
  )

### 6. Seleção de Colunas

In [0]:
gold_cubo_captacao_pl = gold_cubo_captacao_pl\
    .select(
        "cnpj_fundo_classe",
        "ano_mes",
        "pl_ultimo_dia_mes",
        "pl_medio_mes",
        "captacao_bruta_mes",
        "resgate_mes",
        "captacao_liquida_mes",
        "nr_cotistas_ultimo_dia",
        "variacao_cotistas_mes",
        "flag_captacao_negativa"
    )

In [0]:
gold_cubo_captacao_pl = gold_cubo_captacao_pl.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

gold_cubo_captacao_pl.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.gold_cubo_captacao_pl")

In [0]:
display(gold_cubo_captacao_pl)